# Ordered Logistic Regression Results for Adoption Predictors in Rangeland Management (FAIR²) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs in the dataset package.

We list all record sets available in the dataset and their corresponding `@id` values. For each record set, all available fields and their `@id`s are also displayed. This helps in referencing them dynamically in later steps.

In [ ]:
# List all record sets and their fields using @id
record_sets_info = dataset.metadata.record_sets

if not record_sets_info:
    print("No top-level record sets declared in metadata. Inspecting downloadable datasets...")
    for obj in dataset.distributions:
        print(f"- Distribution @id: {obj['@id']}")
    print("\nAlternatively, try loading records directly to discover available sets.")
else:
    for rs in record_sets_info:
        print(f"Record set @id: {rs['@id']}")
        if 'fields' in rs:
            for field in rs['fields']:
                print(f"  - Field @id: {field['@id']} ({field.get('name','')})")
        print()

To assist further, let's try to infer available record sets for `dataset.records()` by inspecting their identifiers:

In [ ]:
# Collect available record set IDs for iteration
record_set_ids = dataset.record_sets
if record_set_ids:
    print("Record sets available (by @id):")
    for rid in record_set_ids:
        print("  ", rid)
else:
    print("No record sets found explicitly in metadata. Attempting to enumerate distributions as record sets.")
    # Try distributions if present
    if hasattr(dataset.metadata, 'distributions'):
        for dist in dataset.metadata.distributions:
            print("Distribution @id:", dist['@id'])

## 3. Data Extraction
Load data from each record set (by `@id`) into a DataFrame for analysis.

Below, we loop through all record sets discovered and extract their data, storing each as a DataFrame in a dictionary keyed by the record set `@id`.

In [ ]:
# Extract all data from each record set into a dictionary of DataFrames

# Automatic detection of available record set IDs (fallback to manual list if needed)
if record_set_ids:
    record_sets_to_load = record_set_ids
else:
    # If dataset.record_sets is empty, you may need to specify record set IDs manually, e.g.:
    record_sets_to_load = []  # Add known record set @id's here as strings if available

dataframes = {}
for rs_id in record_sets_to_load:
    print(f"Loading records for record set @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print("  Loaded with columns:", df.columns.tolist())
        else:
            print("  No records found for", rs_id)
    except Exception as e:
        print(f"  Error loading records for {rs_id}: {e}")

# If we couldn't auto-infer a working record set, prompt user for manual entry
if not dataframes:
    print("No dataframes loaded. Try supplying a known record set @id from earlier cell (if any found).")

# Show preview of the first DataFrame loaded, if any
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nFirst data frame loaded for record set @id: {first_rs}")
    print("Columns:", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
In this section, we perform common data processing steps such as filtering, normalization, and grouping. 
All fields and record sets are referenced strictly by their `@id`.

Before continuing, let’s list numeric fields in the first DataFrame as candidate fields for analysis.

In [ ]:
# Identify numeric columns by their @id for the first available record set
import numpy as np
numeric_field_candidates = []
group_field_candidates = []

if dataframes:
    first_rs = list(dataframes.keys())[0]
    df = dataframes[first_rs]
    for col in df.columns:
        if np.issubdtype(df[col].dropna().dtype, np.number):
            numeric_field_candidates.append(col)
        if 'group' in col.lower() or 'ward' in col.lower() or 'county' in col.lower() or df[col].dtype == 'object':
            group_field_candidates.append(col)
    print(f"Numeric field candidates (by @id): {numeric_field_candidates}")
    print(f"Possible group/categorical fields (by @id): {group_field_candidates}")
else:
    print("No DataFrames loaded to analyze numeric fields.")

Select a numeric field and a group field by specifying their `@id` from the candidates above.

Below, we filter the records for values greater than a chosen threshold, normalize a numeric field, and group by a categorical field, all using `@id` column references.

In [ ]:
if dataframes and numeric_field_candidates:
    record_set_id = first_rs  # Use the first loaded record set
    numeric_field_id = numeric_field_candidates[0]
    threshold = df[numeric_field_id].quantile(0.75)  # 75th percentile as example

    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records from '@id' {record_set_id} with '{numeric_field_id}' > {threshold:.2f}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Choose a group field if available
    group_field_id = group_field_candidates[0] if group_field_candidates else None
    if group_field_id and group_field_id in filtered_df.columns:
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No DataFrame or numeric fields available for EDA.")

## 5. Visualization
Below we visualize the distribution of the normalized numeric field and group means, all referencing columns by their `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_candidates:
    record_set_id = first_rs
    numeric_field_id = numeric_field_candidates[0]
    field_norm = f"{numeric_field_id}_normalized"
    df_to_plot = filtered_df

    # Histogram of normalized values
    plt.figure(figsize=(7, 4))
    sns.histplot(df_to_plot[field_norm], kde=True, bins=20, color='skyblue')
    plt.title(f"Distribution of normalized field: {field_norm} (@id)")
    plt.xlabel(field_norm)
    plt.ylabel("Count")
    plt.show()

    # Bar plot of means by group (if available)
    group_field_id = group_field_candidates[0] if group_field_candidates else None
    if group_field_id and group_field_id in df_to_plot.columns:
        group_means = df_to_plot.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8, 4))
        sns.barplot(data=group_means, x=group_field_id, y=numeric_field_id, palette="viridis")
        plt.xticks(rotation=45)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.tight_layout()
        plt.show()
    else:
        print("No group field available for grouped visualization.")
else:
    print("No numeric data available to visualize.")

## 6. Conclusion
In this notebook, we:
- Loaded and explored the FAIR² dataset metadata and discovered record sets and fields via their `@id`s.
- Dynamically referenced and loaded record sets using `mlcroissant`, extracting tabular data for analysis.
- Applied filtering, normalization, and grouping entirely via Croissant schema-driven `@id` references.
- Visualized distributions and groupwise means for example numeric variables in the dataset.

**Key Next Steps:**
- Explore further field-specific analyses depending on schema definitions.
- Perform statistical modeling or hypothesis testing using the normalized data fields.
- Integrate domain context from the dataset documentation for richer insight.

_All references are strictly via Croissant `@id`s to maintain schema traceability and reproducibility._